# Iris Flower Measurements: A Reproducible Data Workflow

**Author:** Ewald Geschwinde

**Dataset:** [Iris](https://archive.ics.uci.edu/dataset/53/iris) (Fisher, 1936), UCI Machine Learning Repository

This project walks a classic tabular dataset through a complete, reproducible
analysis workflow: ingestion from a local CSV, cleaning with documented and
reusable functions, exploratory analysis, and visualization. The dataset records
four morphological measurements for 150 iris flowers drawn evenly from three
species, and the guiding question is how well those measurements distinguish one
species from another. Every step runs top-to-bottom without manual intervention
and without network access, so a reader can reproduce each number and figure
below from the committed CSV alone.

---

## Contents

1. Environment and imports
2. Data ingestion
3. Data cleaning
4. Exploratory data analysis
5. Visualizations
6. Summary and interpretation

## 1. Environment and imports

Library versions are printed deliberately. A notebook that records the versions
it ran under can be re-created later; one that does not leaves a reader guessing
which API a result depended on. The pinned versions also appear in
`requirements.txt`.

In [6]:
import hashlib
import io
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("matplotlib ", plt.matplotlib.__version__)
print("seaborn    ", sns.__version__)

numpy       2.5.4
pandas      3.0.6
matplotlib  3.11.2
seaborn     0.13.2


### Shared plotting style

The figures later in the notebook share one visual system, defined once here so
no single chart invents its own look. Three choices matter:

- **A fixed categorical palette.** The three species each get a permanent colour
  that never changes between figures, so a reader who learns "setosa is blue" in
  Figure 1 can rely on it in Figure 4. The three hues were checked for
  colour-vision-deficiency separation rather than chosen by eye.
- **Marker shapes as a second channel.** Species is encoded by shape as well as
  colour, so the scatter plots stay readable in greyscale, in print, or to a
  reader with colour-vision deficiency. Colour is never the only carrier of
  meaning.
- **Recessive chrome.** Hairline gridlines and muted axis labels keep the data
  marks, not the scaffolding, as the darkest thing on the page.

In [7]:
SPECIES_ORDER = ["setosa", "versicolor", "virginica"]

# Fixed per-species encoding, reused by every figure in this notebook.
SPECIES_COLORS = {
    "setosa":     "#2a78d6",  # blue
    "versicolor": "#eb6834",  # orange
    "virginica":  "#1baf7a",  # aqua
}
SPECIES_MARKERS = {"setosa": "o", "versicolor": "s", "virginica": "^"}

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"

sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "figure.facecolor":  SURFACE,
    "axes.facecolor":    SURFACE,
    "axes.edgecolor":    GRID,
    "axes.labelcolor":   INK,
    "axes.titlecolor":   INK,
    "axes.titleweight":  "bold",
    "axes.grid":         True,
    "grid.color":        GRID,
    "grid.linewidth":    0.6,
    "grid.linestyle":    "-",
    "xtick.color":       MUTED,
    "ytick.color":       MUTED,
    "savefig.facecolor": SURFACE,
    "figure.dpi":        110,
})

## 2. Data ingestion

`iris.csv` sits beside this notebook and was produced from the UCI distribution
file `iris/iris.data` by adding a header row; no values were altered. The
analysis reads this committed copy rather than downloading the data each time it
runs, which is a deliberate reproducibility choice: a file in the repository
cannot change underneath the analysis, whereas a remote one can be revised or
withdrawn without anyone noticing. Keeping the raw download alongside the CSV
means the provenance of every number stays inspectable, and the cleaning steps in
Section 3 remain visible in code rather than being silently baked into the input.

That choice has a cost, which is that the local copy could silently fall behind
the published dataset. Section 2.1 therefore provides a documented way to check
the local file against the source and refresh it, so the convenience of a
committed file does not become a reason to stop verifying it.

The four measurements are recorded in centimetres:

| Column | Meaning |
|---|---|
| `sepal_length_cm` | Length of the sepal (the outer, leaf-like structure) |
| `sepal_width_cm` | Width of the sepal |
| `petal_length_cm` | Length of the petal |
| `petal_width_cm` | Width of the petal |
| `species` | Iris species label: setosa, versicolor, or virginica |

In [8]:
DATA_PATH = "iris.csv"

MEASUREMENTS = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
]
COLUMN_NAMES = MEASUREMENTS + ["species"]


def load_dataset(path=DATA_PATH):
    """Read the Iris CSV into a DataFrame without modifying any values.

    Parsing is kept separate from cleaning so that the untouched input is
    available for comparison against the cleaned output.

    Parameters
    ----------
    path : str
        Path to the Iris CSV file, which must have a header row.

    Returns
    -------
    pandas.DataFrame
        The dataset exactly as stored on disk.
    """
    return pd.read_csv(path)


df_raw = load_dataset()
df_raw.head()

,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [9]:
print(f"Rows:    {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")
print()
df_raw.info()

Rows:    150
Columns: 5

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sepal_length_cm  150 non-null    float64
 1   sepal_width_cm   150 non-null    float64
 2   petal_length_cm  150 non-null    float64
 3   petal_width_cm   150 non-null    float64
 4   species          150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


### 2.1 Optional: verifying the local copy against the source

The function below re-downloads the published file, checks it against a recorded
SHA-256 digest, and rewrites `iris.csv` from it. **It is deliberately not called
anywhere in this notebook**, so the analysis stays offline and deterministic;
it exists so that the committed copy can be audited on demand rather than
trusted indefinitely.

The digest is recorded rather than merely computed, which is what makes the
check meaningful. If the download does not match, the function refuses to
overwrite the local file instead of quietly replacing good data with something
unverified. Should UCI ever legitimately revise the dataset, the recorded digest
should be updated as a deliberate, reviewable edit rather than the check being
removed.

To run it, execute `refresh_from_source()` in a new cell.

In [10]:
SOURCE_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
)

# SHA-256 of the UCI file that iris.csv was built from.
SOURCE_SHA256 = (
    "6f608b71a7317216319b4d27b4d9bc84e6abd734eda7872b71a458569e2656c0"
)


def refresh_from_source(url=SOURCE_URL, expected_sha256=SOURCE_SHA256,
                        path=DATA_PATH):
    """Re-download the published dataset and rewrite the local CSV from it.

    Not called during the analysis; the notebook runs offline from the committed
    copy. Call this manually to confirm that the local file still matches what
    the UCI repository publishes, or to update it if the published data is
    revised.

    The published file has no header row, so column names are applied before the
    CSV is written. The download is rejected if its digest does not match
    ``expected_sha256``, so a verified local copy is never overwritten by
    unverified bytes.

    Parameters
    ----------
    url : str
        Address of the headerless Iris data file at the UCI repository.
    expected_sha256 : str
        Digest the download must match.
    path : str
        Local CSV to rewrite once the download has been verified.

    Returns
    -------
    pandas.DataFrame
        The freshly downloaded dataset.

    Raises
    ------
    ValueError
        If the download does not match the recorded digest. The local file is
        left untouched.
    """
    with urllib.request.urlopen(url, timeout=30) as response:
        payload = response.read()

    digest = hashlib.sha256(payload).hexdigest()
    if digest != expected_sha256:
        raise ValueError(
            "The published file does not match the recorded checksum.\n"
            f"  expected {expected_sha256}\n"
            f"  received {digest}\n"
            f"Refusing to overwrite '{path}'."
        )

    frame = pd.read_csv(
        io.StringIO(payload.decode("utf-8")),
        header=None,
        names=COLUMN_NAMES,
    )
    frame.to_csv(path, index=False)
    print(f"Verified {len(payload)} bytes from UCI.")
    print(f"Rewrote '{path}' with {len(frame)} rows.")
    return frame

In [11]:
#refresh_from_source()

## 3. Data cleaning

## 4. Exploratory data analysis

## 5. Visualizations

## 6. Summary and interpretation